In [1]:
# Minimal NBO Walkthrough


import numpy as np
import veloxchem as vlx

np.set_printoptions(precision=6, suppress=True)

## Molecule

Load water and print the atom numbers used in NBO output.

In [2]:
water_xyz = '''
O   0.000000   0.000000   0.000000
H   0.758602   0.000000   0.504284
H  -0.758602   0.000000   0.504284
'''

molecule = vlx.Molecule.read_molecule_string(water_xyz, units='angstrom')
labels = molecule.get_labels()
coordinates = molecule.get_coordinates_in_angstrom()

print(f"{'atom':>4} {'x':>12} {'y':>12} {'z':>12}")
for atom_number, (label, xyz) in enumerate(zip(labels, coordinates), start=1):
    print(f'{label + str(atom_number):>4} {xyz[0]:12.6f} {xyz[1]:12.6f} {xyz[2]:12.6f}')

atom            x            y            z
  O1     0.000000     0.000000     0.000000
  H2     0.758602     0.000000     0.504284
  H3    -0.758602     0.000000     0.504284


## SCF

Run the ordinary VeloxChem SCF calculation.

In [3]:
basis = vlx.MolecularBasis.read(molecule, 'sto-3g', ostream=None)
scf = vlx.ScfRestrictedDriver()
scf.ostream.mute()
scf.compute(molecule, basis)

print('converged:', scf.is_converged)
print('energy   :', f'{scf.get_scf_energy():.12f} Eh')

converged: True
energy   : -74.945880784112 Eh


## NBO

Pass the SCF orbitals to the NBO driver.

In [4]:
nbo = vlx.NboDriver()
nbo.ostream.mute()
results = nbo.compute(molecule, basis, scf.mol_orbs)

print('NBO result keys:', ', '.join(sorted(results)))

NBO result keys: alternatives, diagnostics, donor_acceptor_diagnostics, mo_analysis, nao_alpha_density_matrix, nao_atom_map, nao_beta_density_matrix, nao_density_matrix, nao_l_map, nao_overlap_matrix, nao_populations, nao_spin_density_matrix, nao_spin_populations, nao_transform, natural_charges, nbo_candidates, nbo_list, primary, provenance, unpaired_electrons


## Read the Result

Inspect one NAO, the natural charges, and the selected NBOs.

In [5]:
nao_index = 0
atom_index = int(results['nao_atom_map'][nao_index])
population = float(results['nao_populations'][nao_index])
print('first NAO:', f'{labels[atom_index]}{atom_index + 1}', 'population =', f'{population:.6f}')

print('\nnatural charges')
for atom_number, label in enumerate(labels, start=1):
    charge = float(results['natural_charges'][atom_number - 1])
    print(f'  {label}{atom_number}: {charge: .6f}')

print('\nselected NBOs')
for item in results['primary']['nbo_list']:
    atom_text = '-'.join(f'{labels[atom]}{atom + 1}' for atom in item['atoms'])
    print(f"  {item['type']:<4} {atom_text:<8} {item['occupation']:.6f}")

first NAO: O1 population = 2.000000

natural charges
  O1: -0.290747
  H2:  0.145373
  H3:  0.145373

selected NBOs
  CR   O1       2.000000
  LP   O1       2.000000
  LP   O1       2.000000
  BD   O1-H2    2.000000
  BD   O1-H3    2.000000


## Allyl Resonance

Use three user inputs: left pi bond, right pi bond, and the two-structure resonance space.

In [6]:
allyl = vlx.Molecule.read_smiles('[CH2+]C=C')
allyl.set_multiplicity(1)
allyl.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
allyl_basis = vlx.MolecularBasis.read(allyl, 'sto-3g', ostream=None)
allyl_scf = vlx.ScfRestrictedDriver()

allyl_scf.compute(allyl, allyl_basis)



                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

{'eri_thresh': 1e-12,
 'scf_type': 'restricted',
 'scf_energy': np.float64(-114.80021875631537),
 'restart': False,
 'filename': None,
 'scf_history': [{'energy': np.float64(-114.80021857572031),
   'gradient_norm': np.float64(0.0006900586487497877),
   'max_gradient': np.float64(7.094660771617397e-05),
   'diff_density': np.float64(0.0),
   'diff_energy': np.float64(-1.5162392230649857e-06)},
  {'energy': np.float64(-114.80021870222977),
   'gradient_norm': np.float64(0.0002738226621131496),
   'max_gradient': np.float64(6.34561901707635e-05),
   'diff_density': np.float64(0.0004264855877934827),
   'diff_energy': np.float64(-1.2650946246139938e-07)},
  {'energy': np.float64(-114.80021874049329),
   'gradient_norm': np.float64(0.00018522087817458136),
   'max_gradient': np.float64(2.0940605985174362e-05),
   'diff_density': np.float64(0.00022625621000555842),
   'diff_energy': np.float64(-3.8263522128545446e-08)},
  {'energy': np.float64(-114.80021875628405),
   'gradient_norm': np.fl

In [8]:
allyl_nbo = vlx.NboDriver()
allyl_nbo.compute(allyl, allyl_basis, allyl_scf.mol_orbs)



{'nao_transform': array([[ 1.001975, -0.252772,  0.001347,  0.008565, -0.000191, -0.000811,
          0.025891,  0.025988,  0.000938,  0.000304,  0.000786,  0.076713,
          0.      ,  0.000147, -0.115845,  0.009542,  0.00143 , -0.      ,
         -0.026525, -0.      ],
        [ 0.002139,  1.293651, -0.005743, -0.060912, -0.000303, -0.00592 ,
         -0.352992, -0.354325, -0.013788, -0.000043, -0.005789, -0.389254,
         -0.      , -0.017935,  0.583081, -0.064475, -0.011932,  0.      ,
          0.357753,  0.      ],
        [ 0.00138 ,  0.014486,  1.00218 , -0.155751,  0.002451,  0.005022,
          0.000851,  0.000925,  0.026229,  0.000713,  0.000702, -0.00758 ,
         -0.      , -0.005263,  0.025073, -0.202178,  0.032215,  0.      ,
          0.145007, -0.      ],
        [-0.005597, -0.108262,  0.005688,  0.793133, -0.009295,  0.052422,
         -0.013036, -0.013043, -0.352858, -0.01639 , -0.018936,  0.080758,
          0.000002, -0.00843 , -0.338438,  1.151186, -0.504564

In [9]:
allyl_inputs = {
    'left': {'required_pi_bonds': [(1, 2)]},
    'right': {'required_pi_bonds': [(2, 3)]},
    'both': {'allowed_pi_bonds': [(1, 2), (2, 3)]},
}

for name, constraints in allyl_inputs.items():
    allyl_results = allyl_nbo.compute(
        allyl,
        allyl_basis,
        allyl_scf.mol_orbs,
        constraints=constraints,
        options={'include_nra': True},
    )
    print(name)
    for structure in allyl_results['nra']['structures']:
        print(f"  {structure['label']:<8} {structure['nra_weight']:.6f}")

left
  pi:1-2   1.000000
right
  pi:2-3   1.000000
both
  pi:2-3   0.599683
  pi:1-2   0.400317
